# Llama 3B saved throughput audit
CPU-only consistency audit of 69 saved development cases. The audit compares saved summary totals with attempt history and exports truncated records with token evidence. Original records remain unchanged.
The archive is generated under `MyDrive/UQRoute-TC/throughput/llama3b-saved-attempt-audit-v1/`. Model responses are runtime artifacts and are excluded from the source repository. Official correctness scoring is separate.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, hashlib, collections, math, zipfile
source = Path('/content/drive/MyDrive/UQRoute-TC/throughput/llama3b-static-strata-v1')
manifest = json.loads((source / 'manifest.json').read_text())
summary = json.loads((source / 'summary.json').read_text())
assert manifest['model_id'] == 'meta-llama/Llama-3.2-3B-Instruct'
assert manifest['max_tokens'] == 1024
assert len(manifest['case_keys']) == len(set(manifest['case_keys'])) == 69
print('Reading saved Llama 3B records on CPU. No inference will run.')


In [ ]:
expected = set(manifest['case_keys'])
paths = sorted((source / 'cases').glob('*/attempts/*.json'))
assert paths, 'No saved attempt files found; check the Drive folder'
records = [json.loads(p.read_text()) for p in paths]
assert set(r['case_key'] for r in records) == expected, 'Saved case coverage differs'
assert len(records) == len(expected), 'Multiple attempts found; preserve and review their history separately'
assert all(r['status'] not in {'started', 'interrupted'} for r in records)
counts = dict(collections.Counter(r['status'] for r in records))
assert counts == summary['status_counts'], (counts, summary['status_counts'])
assert len(records) == summary['attempt_count']
assert math.isclose(sum(r['elapsed_seconds'] for r in records), summary['measured_attempt_seconds'], rel_tol=1e-9)
for key in ['prompt_tokens', 'completion_tokens', 'total_tokens']:
    known = [r['usage'][key] for r in records if isinstance(r.get('usage'), dict) and r['usage'].get(key) is not None]
    assert sum(known) == summary['known_token_totals'][key]
    assert len(records) - len(known) == summary['missing_usage_attempts'][key]
truncated = [r for r in records if r['status'] == 'truncated']
assert all(r.get('finish_reason') == 'length' for r in truncated)
rows = [{key:r.get(key) for key in ['case_key','source_file','benchmark','base_id','status','finish_reason','usage','elapsed_seconds']} for r in records]
report = {'scope':'CPU-only saved-attempt consistency audit, not official scoring', 'model_id':manifest['model_id'], 'attempt_count':len(records), 'status_counts':counts, 'truncated_by_benchmark':dict(collections.Counter(r['benchmark'] for r in truncated)), 'truncated_by_source_file':dict(collections.Counter(r['source_file'] for r in truncated)), 'rows':rows, 'source_sha256':{str(p.relative_to(source)):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}, 'limitations':['Saved evidence labels are retained; token probability alignment is not recomputed here.','Official correctness is not scored.','All original files remain unchanged.']}
destination = source.parent / 'llama3b-saved-attempt-audit-v1'
destination.mkdir(exist_ok=True)
report_path = destination / 'saved_attempt_audit.json'
report_path.write_text(json.dumps(report, indent=2))
archive = destination / 'llama3b-truncation-audit-v1.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    z.write(report_path, 'saved_attempt_audit.json')
    for i, record in enumerate(truncated, 1):
        z.writestr(f'truncated_attempts/{i:02d}.json', json.dumps(record, indent=2))
print(json.dumps({k:report[k] for k in ['attempt_count','status_counts','truncated_by_benchmark','truncated_by_source_file']}, indent=2))
print('Audit saved:', archive)
from google.colab import files
files.download(str(archive))
